# Client-Product Annual Behavior Analysis
This notebook allows you to explore how specific clients purchase specific products year over year.

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Load Data
df = pd.read_csv('data/master_commodities_clean.csv', low_memory=False)
df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Año'] = df['Fecha'].dt.year

print(f'Dataset loaded with {len(df)} rows.')

Dataset loaded with 91978 rows.


## 1. Data Aggregation
We aggregate the data by Client, Product, and Year.

In [6]:
df_annual = df.groupby(['Id_Cliente', 'Id_Producto', 'Año']).agg({
    'Unidades': 'sum',
    'Valores_H': 'sum',
    'Num.Fact': 'nunique'
}).reset_index()
df_annual.columns = ['Id_Cliente', 'Id_Producto', 'Año', 'Total_Unidades', 'Total_Valor', 'Num_Pedidos']

# Get lists for selectors
clients = sorted(df_annual['Id_Cliente'].unique())

print('Aggregation complete.')

Aggregation complete.


## 2. Interactive Visualization
Use the dropdowns to select a Client and a Product.

In [ ]:
client_dropdown = widgets.Dropdown(options=clients, description='Client ID:')
product_dropdown = widgets.Dropdown(description='Product ID:')

def update_products(*args):
    client_id = client_dropdown.value
    products = sorted(df_annual[df_annual['Id_Cliente'] == client_id]['Id_Producto'].unique())
    product_dropdown.options = products

client_dropdown.observe(update_products, 'value')
update_products() # Initialize

@widgets.interact(client_id=client_dropdown, product_id=product_dropdown)
def plot_behavior(client_id, product_id):
    subset = df_annual[(df_annual['Id_Cliente'] == client_id) & (df_annual['Id_Producto'] == product_id)]
    
    if subset.empty:
        print('No data found for this selection.')
        return
    
    # Ensure all years are present for a better timeline
    all_years = pd.DataFrame({'Año': range(df_annual['Año'].min(), df_annual['Año'].max() + 1)})
    subset = all_years.merge(subset, on='Año', how='left').fillna(0)

    fig = px.bar(subset, x='Año', y='Total_Unidades', 
                 title=f'Annual Units: Client {client_id} - Product {product_id}',
                 labels={'Total_Unidades': 'Units Sold'},
                 text='Total_Unidades',
                 color_discrete_sequence=['#636EFA'])
    
    fig.update_traces(textposition='outside')
    fig.update_layout(xaxis_type='category')
    fig.show()
    
    fig_val = px.line(subset, x='Año', y='Total_Valor', markers=True,
                      title=f'Annual Value (€): Client {client_id} - Product {product_id}',
                      labels={'Total_Valor': 'Value (€)'},
                      color_discrete_sequence=['#EF553B'])
    fig_val.update_layout(xaxis_type='category')
    fig_val.show()
    
    display(subset[['Año', 'Total_Unidades', 'Total_Valor', 'Num_Pedidos']])